In [ ]:
import torch
from ultralytics import YOLO
from torchvision import models
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import os


In [ ]:
# --- Config ---
yolo_model_path = "YOLO8m-Experiments/left_right_signs_train/weights/best.pt"
cnn_model_path = "resnet18_left_right_best.pth"
image_dir = "D:/Desktop/test/road_signs_detection/Data2/images/test"
img_size = 224
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# --- Load YOLO model ---
yolo = YOLO(yolo_model_path)
yolo_classes = yolo.model.names  # dict like {0: 'stop', 1: 'right_left'}

In [ ]:
# --- Load CNN model ---
cnn = models.resnet18(pretrained=False)
cnn.fc = torch.nn.Linear(cnn.fc.in_features, 2)
cnn.load_state_dict(torch.load(cnn_model_path, map_location=device))
cnn = cnn.to(device)
cnn.eval()
cnn_classes = ['left', 'right']

# --- Preprocess for CNN ---
def preprocess_cnn(img_crop):
    img = img_crop.resize((img_size, img_size))
    img = np.array(img) / 255.0
    img = torch.tensor(img, dtype=torch.float).permute(2, 0, 1).unsqueeze(0).to(device)
    return img

In [ ]:
# --- Inference on Images ---
for filename in os.listdir(image_dir):
    if not filename.lower().endswith((".jpg", ".png")):
        continue

    img_path = os.path.join(image_dir, filename)
    image = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    # YOLO detection
    results = yolo(image)

    for box in results[0].boxes:
        class_id = int(box.cls)
        conf = float(box.conf)
        label = yolo_classes[class_id]
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # For turn signs, run CNN classifier
        if label == "right_right":
            crop = image.crop((x1, y1, x2, y2))
            input_tensor = preprocess_cnn(crop)
            with torch.no_grad():
                output = cnn(input_tensor)
                pred_class = cnn_classes[torch.argmax(output).item()]
            final_label = f"{pred_class} ({conf:.2f})"
        else:
            final_label = f"{label} ({conf:.2f})"

        # Draw box and label
        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
        draw.text((x1, y1 - 10), final_label, fill="white")

    # Show result
    image.show()  # or image.save("output/" + filename)
